In [ ]:
from sklearn.neighbors import KDTree
import pandas as pd

In [ ]:
fp = "../data/sba_loans_prepared/sba_loans_num_enc_train.csv"
df_train = pd.read_csv(fp)
loan_status_train = df_train.LoanStatus


In [ ]:
df_train.LoanStatus.value_counts()

In [ ]:
preds = [ c for c in df_train.columns.tolist() if c != "LoanStatus"]
X_train_df = df_train[preds]

In [ ]:
from sklearn.neighbors import KDTree

kdt = KDTree(X_train_df, leaf_size=30, metric='euclidean')

In [ ]:
bad_borr_sel = (df_train.LoanStatus == 1)

In [ ]:
NUM_NBRS = 3
X_bad_borr_df = df_train[bad_borr_sel][preds]

In [ ]:
from sklearn.cluster import DBSCAN
import numpy as np
clustering = DBSCAN(eps=3, min_samples=20).fit(X_bad_borr_df)
clustering.labels_

In [ ]:
labels, counts = np.unique(clustering.labels_, return_counts=True)

In [ ]:
counts

In [ ]:
labels

In [ ]:
labels = clustering.fit_predict(X_bad_borr_df)

In [ ]:
centroids = {}
for cluster_id in clustering.labels_:
    # Get all points belonging to the current cluster
    cluster_points = X_bad_borr_df[labels == cluster_id]
    # Calculate the mean of these points to get the centroid
    centroids[cluster_id] = np.mean(cluster_points, axis=0)



In [ ]:
bdist, bind = kdt.query(X_bad_borr_df, k=NUM_NBRS)

In [ ]:
bind = bind.flatten().tolist() + df_train[bad_borr_sel].index.tolist()

In [ ]:
df_bad_borr_nbrh = df_train[df_train.index.isin(bind)]
bad_borr_ind = df_bad_borr_nbrh.index.tolist()
df_bad_borr_info = pd.DataFrame(bad_borr_ind)
df_bad_borr_info.columns = ["bad_borrower_index"]
fpbb = "../data/sba_loans_prepared/bad_borr_index.csv"
df_bad_borr_info.to_csv(fpbb, index=False)

In [ ]:
df_bad_borr_nbrh.LoanStatus.value_counts()

In [ ]:
df_risk_good = df_train

In [ ]:
cols = df_risk_good.columns.tolist()
cols.remove("LoanStatus")
df_risk_good = df_risk_good[cols]


In [ ]:
good_nbrh = set(df_risk_good.index.tolist())  - set(bad_borr_ind)
good_nbrh = list(good_nbrh)

In [ ]:
good_nbrh_mask = df_risk_good.index.isin(good_nbrh)
bad_nbrh_mask = df_risk_good.index.isin(bad_borr_ind)

In [ ]:
df_risk_good.loc[good_nbrh_mask, "RiskyNbrh"] = 0 
df_risk_good.loc[bad_nbrh_mask, "RiskyNbrh"] = 1 

In [ ]:
from scipy.spatial.distance import euclidean

In [ ]:
df_risk_good["LM1D"] = df_risk_good.apply(lambda x: euclidean(centroids[0].values, x[preds]), axis=1)


In [ ]:
df_risk_good["LM2D"] = df_risk_good.apply(lambda x: euclidean(centroids[-1].values, x[preds]), axis=1)


In [ ]:
df_risk_good.RiskyNbrh.value_counts()

In [ ]:
fptrain = "../data/sba_loans_prepared/sba_train_raw.csv"
dftrain_raw = pd.read_csv(fptrain)
dftrain_raw.LoanStatus.value_counts()

In [ ]:
df_bad_borr_raw = dftrain_raw[df_train.index.isin(bind)]
fprrn = "../data/sba_loans_prepared/sba_train_raw_risky_nbrh.csv"
df_bad_borr_raw.to_csv(fprrn, index=False)

In [ ]:
df_bad_borr_raw.LoanStatus.value_counts()

In [ ]:
df_risk_good["LoanStatus"] = loan_status_train
sel_good_nbrh = df_risk_good.RiskyNbrh == 0
df_risk_good[sel_good_nbrh].LoanStatus.value_counts()

In [ ]:
sel_bad_nbrh = df_risk_good.RiskyNbrh == 1
df_risk_good[sel_bad_nbrh].LoanStatus.value_counts()

**NOTE:** The occurence of charge offs in the the good neighborhood (not risky, RiskyNbrh=0) is 0. So if x belongs to a good neighborhood, it cannot be associated with a charge off.

In [ ]:
# this is the training set
fp_rg = "../data/sba_loans_prepared/sba_loans_risk_good_train.csv"
df_risk_good.to_csv(fp_rg, index=False)

In [ ]:
df_bad_borr_nbrh.LoanStatus.value_counts()

In [ ]:
fp = "../data/sba_loans_prepared/sba_loans_num_enc_val.csv"
df_val = pd.read_csv(fp)
X_val = df_val[preds]

In [ ]:
vdist, vind = kdt.query(X_val, k=1)

In [ ]:
import numpy as np
val_nn_dist = {"nn_idx": vind.flatten()}
df_val_res = pd.DataFrame.from_dict(val_nn_dist, orient="columns")

In [ ]:
df_val_res["RiskyNbrh"] = df_val_res.nn_idx.isin(bad_borr_ind)

In [ ]:
df_val_res["RiskyNbrh"] = df_val_res["RiskyNbrh"].map({False: 0, True:1})

In [ ]:
df_val["LM1D"] = df_val.apply(lambda x: euclidean(centroids[0].values, x[preds]), axis=1)
df_val["LM2D"] = df_val.apply(lambda x: euclidean(centroids[-1].values, x[preds]), axis=1)

In [ ]:
df_val["RiskyNbrh"] = df_val_res["RiskyNbrh"]
fp =  "../data/sba_loans_prepared/sba_loans_num_enc_val.csv"
df_val.to_csv(fp, index=False)

In [ ]:
pp_val = df_val_res.RiskyNbrh.value_counts()

In [ ]:
pp_val[1]/pp_val[0]

In [ ]:
fp =  "../data/sba_loans_prepared/sba_loans_num_enc_test.csv"
df_test = pd.read_csv(fp)

In [ ]:
X_test = df_test[preds]

In [ ]:
tdist, tind = kdt.query(X_test, k=1)

In [ ]:
test_nn_dist = {"nn_idx": tind.flatten()}
df_test_res = pd.DataFrame.from_dict(test_nn_dist, orient="columns")

In [ ]:
df_test_res["RiskyNbrh"] = df_test_res.nn_idx.isin(bad_borr_ind)

In [ ]:
df_test["LM1D"] = df_test.apply(lambda x: euclidean(centroids[0].values, x[preds]), axis=1)
df_test["LM2D"] = df_test.apply(lambda x: euclidean(centroids[-1].values, x[preds]), axis=1)

In [ ]:
df_test["RiskyNbrh"] = df_test_res["RiskyNbrh"]
fp =  "../data/sba_loans_prepared/sba_loans_num_enc_test.csv"
df_test.to_csv(fp, index=False)

In [ ]:
df_val